In [1]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
#df = pd.read_csv('/Users/visjostr/school/parking-prediction/parking-data/Turkuparkkihub_parkings_anon.csv', delimiter=';')
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime objects with specified format
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M')

# Define the time for which you want to check the entries
target_time = pd.to_datetime('03/12/2021 12.00', format='%d/%m/%Y %H.%M')  # Example time, adjust as needed

# Define the parking space ID you want to analyze
parking_space_id = '4b727054-c4dc-492e-8303-0fca60acf0ea'  # Example parking space ID, replace with the actual ID you want to analyze

# Filter the DataFrame for the specific parking space ID
parking_space_df = df[df['region_id'] == parking_space_id]

print(parking_space_df[(parking_space_df['time_start'] <= target_time) & (target_time <= parking_space_df['time_end'])])

# Check the entries at the specified time for the specific parking space
entries_at_time = ((parking_space_df['time_start'] <= target_time) & (target_time <= parking_space_df['time_end'])).sum()

print("Number of entries at", target_time, "for parking space", parking_space_id, ":", entries_at_time)

              created_at       modified_at  \
327018  01/12/2021 16.11  03/12/2021 12.49   
331236  02/12/2021 12.42  03/12/2021 13.50   
332964  02/12/2021 15.16  03/12/2021 12.05   
333005  02/12/2021 15.22  03/12/2021 12.11   
333046  02/12/2021 15.23  03/12/2021 12.44   
...                  ...               ...   
337701  03/12/2021 11.54  03/12/2021 12.30   
337784   03/12/2021 9.36  03/12/2021 12.37   
337786  03/12/2021 11.33  03/12/2021 12.37   
337867  03/12/2021 11.46  03/12/2021 12.45   
338565  03/12/2021 11.58  03/12/2021 13.41   

                                          id  \
327018  7aea6c90-ca7e-422f-b5c8-410b7408ff64   
331236  e2774200-25c3-4f5d-904e-767e0f410beb   
332964  f2348360-8aa8-483f-b36a-5a70801c7208   
333005  c43497e3-72f2-445f-8908-ea85cbfdd605   
333046  ffcd9ec2-9100-4712-b138-1f312b9a24be   
...                                      ...   
337701  cd10df24-4355-4fad-9a71-4ed969214cfb   
337784  80b20710-1b91-453e-bcbf-276f22ebfb35   
337786  2649e6d

In [2]:
import pandas as pd
from datetime import datetime, timedelta

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 9, 1)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest 15 minutes
def round_to_15_minutes(dt):
    reference_date = datetime(2000, 1, 1)  # Choose a reference date for rounding
    rounded = dt - reference_date
    rounded = rounded + timedelta(minutes=7.5)  # Add half interval to round correctly
    rounded -= rounded % timedelta(minutes=15)  # Round to nearest 15 minutes
    return rounded + reference_date

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_15_minutes(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=15)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series.csv', index=False)

# Print time series DataFrame
print(time_series_df)


                     Time  Parkings
156   2021-09-01 02:30:00         1
157   2021-09-01 02:45:00         1
158   2021-09-01 03:00:00         1
159   2021-09-01 03:15:00         1
53    2021-09-01 03:30:00         2
...                   ...       ...
13042 2022-01-14 23:00:00         4
13043 2022-01-14 23:15:00         3
13044 2022-01-14 23:30:00         1
13045 2022-01-14 23:45:00         1
13046 2022-01-15 00:00:00         1

[13047 rows x 2 columns]


In [2]:
import pandas as pd
from datetime import datetime, timedelta

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest minute
def round_to_minute(dt):
    return dt - timedelta(seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_minute(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=1)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_minutes_three.csv', index=False)

# Print time series DataFrame
print(time_series_df)


                      Time  Parkings
26724  2021-10-11 00:10:00         1
26725  2021-10-11 00:11:00         1
26726  2021-10-11 00:12:00         1
26727  2021-10-11 00:13:00         1
26728  2021-10-11 00:14:00         1
...                    ...       ...
138226 2022-01-14 23:56:00         1
138227 2022-01-14 23:57:00         1
138228 2022-01-14 23:58:00         1
138229 2022-01-14 23:59:00         1
138230 2022-01-15 00:00:00         1

[138231 rows x 2 columns]


In [4]:
import pandas as pd
from datetime import datetime, timedelta

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest minute
def round_to_minute(dt):
    return dt - timedelta(seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']
    zone_id = row['zone_id']  # Extracting zone_id for each row

    current_time = round_to_minute(time_start)
    while current_time <= time_end:
        key = (current_time, zone_id)  # Using a tuple of (time, zone_id) as key
        if key not in time_series:
            time_series[key] = 0
        time_series[key] += 1
        current_time += timedelta(minutes=1)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time_Zone', 'Parkings'])

# Splitting 'Time_Zone' column into 'Time' and 'Zone_id'
time_series_df[['Time', 'Zone_id']] = pd.DataFrame(time_series_df['Time_Zone'].tolist(), index=time_series_df.index)
time_series_df.drop(columns=['Time_Zone'], inplace=True)

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Export DataFrame to a CSV file
#time_series_df.to_csv('parking_time_series_minutes_three.csv', index=False)

# Print time series DataFrame
print(time_series_df)


        Parkings                Time  Zone_id
44878          1 2021-10-11 00:10:00        2
44879          1 2021-10-11 00:11:00        2
44880          1 2021-10-11 00:12:00        2
44881          1 2021-10-11 00:13:00        2
44882          1 2021-10-11 00:14:00        2
...          ...                 ...      ...
551586         1 2022-01-14 23:56:00        2
551587         1 2022-01-14 23:57:00        2
551588         1 2022-01-14 23:58:00        2
551589         1 2022-01-14 23:59:00        2
551590         1 2022-01-15 00:00:00        2

[551643 rows x 3 columns]


In [3]:


#print(df.terminal_number.mode())
#print(df.zone_id.nunique())
#print(df[df['zone_id'] == 1].region_id.unique())
print(df[df['is_disc_parking'] == '1'].is_disc_parking)
print(df.is_disc_parking.nunique())

#print(df[df['parking_area_id'] == '52c37d3d-2a28-4bde-8996-fdff25e4157d'])

318815    1
Name: is_disc_parking, dtype: object
2
